In [3]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

load_dotenv()

# 日本語フォントの設定
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# パス設定
BASE_DIR = Path("/app")
INSERT_DIR = BASE_DIR / "data" / "raw" / "package_inserts"
GUIDELINE_DIR = BASE_DIR / "data" / "raw" / "guidelines"
CHROMA_BASE_DIR = BASE_DIR / "data" / "chroma_db"

# OpenAI設定
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

# プロンプトテンプレート
prompt_template = """
あなたは医薬品の専門家です。
以下の医薬品添付文書およびガイドラインの情報をもとに質問に回答してください。

提供された情報に含まれていない内容については「提供された文書には記載がありません」と答えてください。
提供された文書に記載がない内容は、一般的な知識であっても回答しないでください。
回答の最後に必ず参照した文書名を記載してください。

【参照情報】
{context}

【質問】
{question}

【回答】
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("共通設定完了")

共通設定完了


In [4]:
# PDF読み込み関数
def load_pdfs_with_metadata(pdf_dir: Path, doc_type: str) -> list:
    all_documents = []
    for pdf_path in sorted(pdf_dir.glob("*.pdf")):
        loader = PyMuPDFLoader(str(pdf_path))
        pages = loader.load()
        for page in pages:
            page.metadata["doc_type"] = doc_type
            page.metadata["file_name"] = pdf_path.name
            page.metadata["drug_name"] = pdf_path.stem
        all_documents.extend(pages)
    return all_documents

# 評価関数
def evaluate_response(question_data: dict, rag_chain) -> dict:
    question = question_data["question"]
    relevant_keywords = question_data["relevant_keywords"]

    result = rag_chain.invoke({"query": question})
    answer = result["result"]
    source_docs = result["source_documents"]

    matched_keywords = [kw for kw in relevant_keywords if kw in answer]
    keyword_score = len(matched_keywords) / len(relevant_keywords)

    retrieved_files = list(set([
        doc.metadata.get("file_name") for doc in source_docs
    ]))
    relevant_docs = question_data["relevant_docs"]

    precision = len([f for f in retrieved_files if f in relevant_docs]) / len(retrieved_files) if retrieved_files else 0
    recall = len([f for f in retrieved_files if f in relevant_docs]) / len(relevant_docs) if relevant_docs else 0
    f1 = (2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0)

    return {
        "id": question_data["id"],
        "category": question_data["category"],
        "question": question,
        "answer": answer,
        "keyword_score": round(keyword_score, 3),
        "matched_keywords": matched_keywords,
        "unmatched_keywords": [kw for kw in relevant_keywords if kw not in matched_keywords],
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "retrieved_files": retrieved_files,
        "relevant_docs": relevant_docs,
    }

# 評価用質問セット
eval_questions = [
    {
        "id": "Q01", "category": "禁忌",
        "question": "メトホルミンの禁忌を教えてください",
        "relevant_docs": ["metformin.pdf"],
        "relevant_keywords": [
            "乳酸アシドーシス", "重度の腎機能障害", "eGFR", "30ml/min/1.73m2未満",
            "重度の肝機能障害", "低酸素血症", "脱水", "重症ケトーシス", "糖尿病性昏睡",
            "1型糖尿病", "重症感染症", "手術", "妊娠", "過敏症", "アルコール"
        ]
    },
    {
        "id": "Q02", "category": "用法用量",
        "question": "グリメピリドの用法・用量を教えてください",
        "relevant_docs": ["glimepiride.pdf"],
        "relevant_keywords": [
            "0.5mg", "1mg", "1日1～2回", "朝", "食前", "食後",
            "維持量", "1～4mg", "最高投与量", "6mg"
        ]
    },
    {
        "id": "Q03", "category": "副作用",
        "question": "エンパグリフロジンの主な副作用を教えてください",
        "relevant_docs": ["empagliflozin.pdf"],
        "relevant_keywords": [
            "低血糖", "脱水", "ケトアシドーシス", "腎盂腎炎",
            "尿路感染症", "膀胱炎", "頻尿", "多尿", "口渇", "体重減少"
        ]
    },
    {
        "id": "Q04", "category": "禁忌",
        "question": "オゼンピックの禁忌を教えてください",
        "relevant_docs": ["semaglutide.pdf"],
        "relevant_keywords": [
            "過敏症", "糖尿病性ケトアシドーシス", "糖尿病性昏睡",
            "1型糖尿病", "重症感染症", "手術"
        ]
    },
    {
        "id": "Q05", "category": "薬剤比較",
        "question": "腎機能が低下している患者に使用できる糖尿病薬は何ですか",
        "relevant_docs": ["linagliptin.pdf", "empagliflozin.pdf"],
        "relevant_keywords": [
            "腎機能障害", "eGFR", "リナグリプチン", "用量調節不要",
            "エンパグリフロジン", "20mL/min/1.73m2"
        ]
    },
    {
        "id": "Q06", "category": "薬剤比較",
        "question": "DPP-4阻害薬のジャヌビアとトラゼンタの腎機能低下時の使い分けを教えてください",
        "relevant_docs": ["sitagliptin.pdf", "linagliptin.pdf"],
        "relevant_keywords": [
            "シタグリプチン", "50mg", "12.5mg", "CrCl", "30未満",
            "リナグリプチン", "5mg", "用量調節"
        ]
    },
    {
        "id": "Q07", "category": "薬剤比較",
        "question": "インスリン グラルギンとインスリン アスパルトの違いを教えてください",
        "relevant_docs": ["insulin_glargine.pdf", "insulin_aspart.pdf"],
        "relevant_keywords": ["持効型", "超速効型", "作用発現時間", "10〜20分"]
    },
    {
        "id": "Q08", "category": "ガイドライン",
        "question": "糖尿病の血糖コントロール目標を教えてください",
        "relevant_docs": ["diabetes_manual_2025.pdf"],
        "relevant_keywords": [
            "HbA1c", "7.0%未満", "グリコアルブミン", "20%未満", "6.0%", "8.0%"
        ]
    },
    {
        "id": "Q09", "category": "ガイドライン",
        "question": "2型糖尿病の薬物療法の第一選択薬は何ですか",
        "relevant_docs": ["diabetes_manual_2025.pdf"],
        "relevant_keywords": [
            "ビグアナイド", "単剤で開始", "ステップ1",
            "eGFR", "30ml/分/1.73m2", "食事療法", "運動療法"
        ]
    },
    {
        "id": "Q10", "category": "横断検索",
        "question": "心血管疾患を合併した2型糖尿病患者に推奨される薬剤は何ですか",
        "relevant_docs": ["empagliflozin.pdf", "semaglutide.pdf", "diabetes_manual_2025.pdf"],
        "relevant_keywords": [
            "SGLT2阻害薬", "心血管疾患", "心不全",
            "微量アルブミン尿", "肥満", "1剤上乗せ", "ステップ2"
        ]
    },
]

print(f"PDF読み込み関数・評価関数・質問セット定義完了")
print(f"評価問題数: {len(eval_questions)}問")

PDF読み込み関数・評価関数・質問セット定義完了
評価問題数: 10問


In [12]:
# ===== 検証用セル（開発時の問題調査） =====
# sitagliptin.pdfの未検出チャンクの内容を確認
sitagliptin_no_section = [
    c for c in section_chunks
    if c.metadata.get("file_name") == "sitagliptin.pdf"
    and not c.metadata.get("section")
]

print(f"sitagliptin.pdf 未検出チャンク数: {len(sitagliptin_no_section)}")
print("\n--- 先頭5チャンクの内容（repr表示）---")
for i, chunk in enumerate(sitagliptin_no_section[:5]):
    print(f"\nチャンク{i+1}:")
    print(repr(chunk.page_content[:200]))

sitagliptin.pdf 未検出チャンク数: 28

--- 先頭5チャンクの内容（repr表示）---

チャンク1:
'－1－\n禁忌（次の患者には投与しないこと）\n本剤の成分に対し過敏症の既往歴のある患者\n重症ケトーシス、糖尿病性昏睡又は前昏睡、1型糖尿病\nの患者［輸液及びインスリンによる速やかな高血糖の是\n正が必須となるので本剤を投与すべきでない。］\n重症感染症、手術前後、重篤な外傷のある患者［インス\nリン注射による血糖管理が望まれるので本剤の投与は適\nさない。］\n組成・性状\n組成\n販売名\nジャヌビア®\n錠12.'

チャンク2:
'三二酸化鉄、\n黒酸化鉄\n結晶セルロース、無水リン酸水素カルシ\nウム、クロスカルメロースナトリウム、\nステアリン酸マグネシウム、フマル酸ス\nテアリルナトリウム、没食子酸プロピル、\nポリビニルアルコール（部分けん化物）、\n酸化チタン、マクロゴール4000、タル\nク、黄色三二酸化鉄、三二酸化鉄\n製剤の性状\n販売名\nジャヌビア®\n錠12.5mg\nジャヌビア®\n錠25mg\nジャヌビア®\n錠50mg\nジャヌビ'

チャンク3:
'重量\n約104mg\n約105mg\n約209mg\n約416mg\n識別コード\nMSD 211\nMSD 221\nMSD 112\nMSD 277\n効能又は効果\n2型糖尿病\n効能又は効果に関連する注意\n本剤の適用はあらかじめ糖尿病治療の基本である食事療法、運\n動療法を十分に行ったうえで効果が不十分な場合に限り考慮す\nること。\n用法及び用量\n通常、成人にはシタグリプチンとして50mgを1日1回経口投与\nする。'

チャンク4:
'50mg\n1日1回\n重度、末\n期腎不全\nCrCl<30\n男性：Cr>2.5\n女性：Cr>2.0\n12.5mg\n1日1回\n25mg\n1日1回\n*\u200c\x07クレアチニンクリアランスに概ね相当する値\n末期腎不全患者については、血液透析との時間関係は問わな\nい。［9.2.1、9.8、16.6.1 参照］\n2.\n2.1\n2.2\n2.3\n3.\n3.1\n＊\n3.2\n4.\n5.\n6.\n7.\n7.1\n7.2\n2025年1'

チャンク5:
'－2－\n重要

In [21]:
# ガイドライン参照が必要な質問を判定するキーワード
# ガイドラインのセクション分割実装時は合わせて見直す
GUIDELINE_KEYWORDS = [
    "ガイドライン", "推奨", "第一選択", "治療方針",
    "血糖コントロール", "目標", "標準治療"
]


def detect_sections(question: str, llm) -> list:
    """
    質問文から参照すべきセクション番号をLLMに判定させる
    """
    # 質問パターン別の参照セクション定義
    question_patterns = {
        "禁忌": [1, 2, 8, 10, 14, 16],
        "用法用量": [6, 7, 8, 9],
        "副作用": [1, 8, 9, 11],
        "腎機能": [1, 2, 6, 7, 8, 9, 14, 16],
        "妊婦": [2, 9],
        "相互作用": [1, 2, 8, 10],
        "薬物動態": [8, 9, 13],
        "作用機序": [15, 18],
    }

    pattern_descriptions = """
- 禁忌：投与してはいけない患者・条件に関する質問
- 用法用量：投与方法・投与量・投与回数に関する質問
- 副作用：副作用・有害事象に関する質問
- 腎機能：腎機能障害患者への投与・用量調節に関する質問
- 妊婦：妊婦・授乳婦・生殖への影響に関する質問
- 相互作用：他の薬剤との併用・相互作用に関する質問
- 薬物動態：吸収・分布・代謝・排泄に関する質問
- 作用機序：薬効・作用機序に関する質問
"""

    prompt = f"""
以下は医薬品に関する質問パターンの一覧です。

{pattern_descriptions}

次の質問が該当するパターンを全て選んでください。
パターン名のみをカンマ区切りで答えてください。
例：禁忌, 腎機能

質問：{question}

該当するパターン：
"""

    response = llm.invoke(prompt)
    response_text = response.content.strip()

    # 該当パターンからセクション番号を収集
    section_numbers = []
    for pattern_name, numbers in question_patterns.items():
        if pattern_name in response_text:
            for num in numbers:
                if num not in section_numbers:
                    section_numbers.append(num)

    is_guideline_query = any(kw in question for kw in GUIDELINE_KEYWORDS)
    return section_numbers, response_text, is_guideline_query


def get_section_keywords(section_numbers: list) -> list:
    """
    セクション番号からChromaDBフィルタ用キーワードリストを作る
    セクションメタデータは「2. 禁忌」のような形式で保存されている
    """
    keywords = [f"{num}." for num in section_numbers]
    return keywords


# 動作確認（評価質問セットの全問でテスト）
print("=== セクション判定テスト ===\n")
for q in eval_questions:
    section_numbers, raw_response, is_guideline = detect_sections(q["question"], llm)
    keywords = get_section_keywords(section_numbers)
    print(f"{q['id']} [{q['category']}] {q['question'][:25]}...")
    print(f"  LLM判定：{raw_response}")
    print(f"  セクション番号：{section_numbers}")
    print(f"  フィルタキーワード：{keywords}")
    print(f"  ガイドラインフォールバック：{is_guideline}")
    print()

=== セクション判定テスト ===

Q01 [禁忌] メトホルミンの禁忌を教えてください...
  LLM判定：禁忌
  セクション番号：[1, 2, 8, 10, 14, 16]
  フィルタキーワード：['1.', '2.', '8.', '10.', '14.', '16.']
  ガイドラインフォールバック：False

Q02 [用法用量] グリメピリドの用法・用量を教えてください...
  LLM判定：用法用量
  セクション番号：[6, 7, 8, 9]
  フィルタキーワード：['6.', '7.', '8.', '9.']
  ガイドラインフォールバック：False

Q03 [副作用] エンパグリフロジンの主な副作用を教えてください...
  LLM判定：副作用
  セクション番号：[1, 8, 9, 11]
  フィルタキーワード：['1.', '8.', '9.', '11.']
  ガイドラインフォールバック：False

Q04 [禁忌] オゼンピックの禁忌を教えてください...
  LLM判定：禁忌
  セクション番号：[1, 2, 8, 10, 14, 16]
  フィルタキーワード：['1.', '2.', '8.', '10.', '14.', '16.']
  ガイドラインフォールバック：False

Q05 [薬剤比較] 腎機能が低下している患者に使用できる糖尿病薬は何で...
  LLM判定：腎機能, 用法用量
  セクション番号：[6, 7, 8, 9, 1, 2, 14, 16]
  フィルタキーワード：['6.', '7.', '8.', '9.', '1.', '2.', '14.', '16.']
  ガイドラインフォールバック：False

Q06 [薬剤比較] DPP-4阻害薬のジャヌビアとトラゼンタの腎機能低...
  LLM判定：腎機能, 用法用量
  セクション番号：[6, 7, 8, 9, 1, 2, 14, 16]
  フィルタキーワード：['6.', '7.', '8.', '9.', '1.', '2.', '14.', '16.']
  ガイドラインフォールバック：False

Q07 [薬剤比較] インスリン グラルギンとインスリン アスパルトの違...
  LLM判定：作用機序, 薬物動

In [35]:
def detect_drug_names(question: str, llm) -> list:
    """
    質問文から薬剤名（ファイル名のstem）をLLMに抽出させる
    """
    drug_map = """
    - メトホルミン、メトグルコ → metformin
    - グリメピリド、アマリール → glimepiride
    - エンパグリフロジン、ジャディアンス → empagliflozin
    - セマグルチド、オゼンピック、リベルサス → semaglutide
    - シタグリプチン、ジャヌビア → sitagliptin
    - リナグリプチン、トラゼンタ → linagliptin
    - インスリン グラルギン、ランタス → insulin_glargine
    - インスリン アスパルト、ノボラピッド → insulin_aspart
    """

    prompt = f"""
以下は薬剤名とファイル名の対応表です。
{drug_map}

次の質問に登場する薬剤のファイル名を全て答えてください。
ファイル名のみをカンマ区切りで答えてください。
特定の薬剤名がない場合は「なし」と答えてください。
例：metformin, glimepiride

質問：{question}

ファイル名：
"""

    response = llm.invoke(prompt)
    response_text = response.content.strip()

    if "なし" in response_text:
        return []

    drug_names = [
        name.strip()
        for name in response_text.replace("、", ",").split(",")
        if name.strip()
    ]
    return drug_names

In [36]:
# 薬剤名抽出のテスト

# ===== 検証用セル②：薬剤名抽出関数の動作確認（全10問） =====
# 目的：detect_drug_names()が質問文から正しく薬剤名を抽出できるか確認
# 確認ポイント：特定薬剤を含まない質問（Q05・Q08・Q10）が
#              空リストで返ることを確認

for q in eval_questions:
    drug_names = detect_drug_names(q["question"], llm)
    print(f"{q['id']}: {drug_names}")

Q01: ['metformin']
Q02: ['glimepiride']
Q03: ['empagliflozin']
Q04: ['semaglutide']
Q05: []
Q06: ['sitagliptin', 'linagliptin']
Q07: ['insulin_glargine', 'insulin_aspart']
Q08: []
Q09: ['metformin', 'glimepiride']
Q10: []


In [33]:
# Q04のデバッグ

# ===== 検証用セル①：セクションフィルタの動作確認（Q04） =====
# 問題：セクション番号でフィルタしているにも関わらず
#       関係ない薬剤のチャンクが混入していた
# 発見：section_numberメタデータがない旧形式PDFのチャンクが
#       他の薬剤のセクションとして誤ってヒットしていた

question = "オゼンピックの禁忌を教えてください"
section_numbers, _, is_guideline = detect_sections(question, llm)

print(f"セクション番号：{section_numbers}")

retrieved_docs = []
for num in section_numbers:
    results = vectorstore_c.similarity_search(
        question,
        k=2,
        filter={
            "$and": [
                {"doc_type": {"$eq": "package_insert"}},
                {"section_number": {"$eq": str(num)}}
            ]
        }
    )
    print(f"セクション{num}：{len(results)}件")
    for r in results:
        print(f"  → {r.metadata.get('file_name')} section_number={r.metadata.get('section_number')}")
    retrieved_docs.extend(results)

print(f"\n合計取得チャンク数：{len(retrieved_docs)}")
print(f"フォールバック発動：{not retrieved_docs}")

セクション番号：[1, 2, 8, 10, 14, 16]
セクション1：2件
  → glimepiride.pdf section_number=1
  → metformin.pdf section_number=1
セクション2：2件
  → insulin_aspart.pdf section_number=2
  → insulin_glargine.pdf section_number=2
セクション8：2件
  → insulin_glargine.pdf section_number=8
  → insulin_glargine.pdf section_number=8
セクション10：2件
  → glimepiride.pdf section_number=10
  → insulin_glargine.pdf section_number=10
セクション14：2件
  → empagliflozin.pdf section_number=14
  → semaglutide.pdf section_number=14
セクション16：2件
  → insulin_glargine.pdf section_number=16
  → empagliflozin.pdf section_number=16

合計取得チャンク数：12
フォールバック発動：False


In [38]:
# Q08のデバッグ
# ===== 検証用セル③：ガイドライン検索の単独動作確認（Q08） =====
# 目的：evaluate_response_dで0点になっている原因調査の第一段階
# 確認：ガイドライン検索自体は正常に3件取得できることを確認
# 結論：問題はガイドライン検索ではなくevaluate_response_d内の処理にある

question = "糖尿病の血糖コントロール目標を教えてください"
section_numbers, _, is_guideline = detect_sections(question, llm)
print(f"is_guideline: {is_guideline}")
print(f"section_numbers: {section_numbers}")

# ガイドライン検索を単独で実行
guideline_results = vectorstore_c.similarity_search(
    question,
    k=3,
    filter={"doc_type": {"$eq": "guideline"}}
)

print(f"\nガイドライン検索結果: {len(guideline_results)}件")

for r in guideline_results:
    print(f"  → {r.metadata.get('file_name')}")
    print(f"     {r.page_content[:100]}")

is_guideline: True
section_numbers: []

ガイドライン検索結果: 3件
  → diabetes_manual_2025.pdf
     糖尿病標準診療マニュアル2025
14
on cardiovascular outcomes and mortality in type 2 diabetes
（J-DOIT3）: an open-l
  → diabetes_manual_2025.pdf
     7
糖尿病標準診療マニュアル2025
薬剤選択は血管合併症・低血糖に関するエビデンスの有無やコストなどにより判断した．
3 〜6 ヵ月ごとに患者の病態や目標値を見直す．
薬物療法はステップ1 から開始
  → diabetes_manual_2025.pdf
     5
糖尿病標準診療マニュアル2025
血糖	 HbA1c 	
7.0% 未満
B, C, 40 ─42（グリコアルブミン
 43, 44 約20％未満）
	
空腹時血糖	
130 mg／dl 未満
脂


In [40]:
# Q08を単独でデバッグ
# ===== 検証用セル④：evaluate_response_d内部のステップ確認（Q08） =====
# 目的：③でガイドライン検索自体は正常と判明したため
#       evaluate_response_d関数内の処理を1ステップずつ追って原因を特定
# 発見：section_numbersが空のときフォールバックが発動し
#       関係ない添付文書5件でretrieved_docsが埋まっていた
#       その結果ガイドラインが5件の枠に入れなかった
# 対応：if section_numbers and not retrieved_docs: に修正

question = "糖尿病の血糖コントロール目標を教えてください"
section_numbers, _, is_guideline = detect_sections(question, llm)
drug_names = detect_drug_names(question, llm)

print(f"section_numbers: {section_numbers}")
print(f"is_guideline: {is_guideline}")
print(f"drug_names: {drug_names}")

retrieved_docs = []

if section_numbers:
    print("セクション検索実行")
else:
    print("セクション検索スキップ")

if is_guideline:
    print("ガイドライン検索実行")
    guideline_results = vectorstore_c.similarity_search(
        question,
        k=3,
        filter={"doc_type": {"$eq": "guideline"}}
    )
    print(f"ガイドライン結果: {len(guideline_results)}件")
    for r in guideline_results:
        print(f"  → {r.metadata.get('file_name')}")
    retrieved_docs.extend(guideline_results)

print(f"\nretrieved_docs合計: {len(retrieved_docs)}")
print(f"retrieved_files: {list(set([d.metadata.get('file_name') for d in retrieved_docs]))}")
print(f"relevant_docs: {eval_questions[7]['relevant_docs']}")

section_numbers: []
is_guideline: True
drug_names: []
セクション検索スキップ
ガイドライン検索実行
ガイドライン結果: 3件
  → diabetes_manual_2025.pdf
  → diabetes_manual_2025.pdf
  → diabetes_manual_2025.pdf

retrieved_docs合計: 3
retrieved_files: ['diabetes_manual_2025.pdf']
relevant_docs: ['diabetes_manual_2025.pdf']


In [42]:
# Q10のデバッグ
# ===== 検証用セル⑤：セクション判定と薬剤名抽出の確認（Q10） =====
# 目的：Q10が0点のままの原因調査
# 確認：section_numbersが11個と多すぎることとdrug_namesが空であることを確認
# 結論：薬剤名が特定できないため添付文書チャンクが枠を埋めてしまう問題が発覚
#       → 次のセル（⑥）で取得チャンクの中身を確認

question = "心血管疾患を合併した2型糖尿病患者に推奨される薬剤は何ですか"
section_numbers, raw_response, is_guideline = detect_sections(question, llm)
drug_names = detect_drug_names(question, llm)

print(f"LLM判定：{raw_response}")
print(f"section_numbers: {section_numbers}")
print(f"is_guideline: {is_guideline}")
print(f"drug_names: {drug_names}")

LLM判定：禁忌, 用法用量, 妊婦, 相互作用, 作用機序
section_numbers: [1, 2, 8, 10, 14, 16, 6, 7, 9, 15, 18]
is_guideline: True
drug_names: []


In [43]:
print(eval_questions[9]["relevant_docs"])
print(eval_questions[9]["relevant_keywords"])

['empagliflozin.pdf', 'semaglutide.pdf', 'diabetes_manual_2025.pdf']
['SGLT2阻害薬', '心血管疾患', '心不全', '微量アルブミン尿', '肥満', '1剤上乗せ', 'ステップ2']


In [44]:
# Q10の取得チャンク確認
# ===== 検証用セル⑥：取得チャンクの中身確認（Q10） =====
# 目的：⑤でsection_numbersが11個・drug_namesが空と判明したため
#       実際に何のチャンクが取得されているか中身を確認
# 発見：添付文書チャンクが先に5件の枠を埋めてしまい
#       ガイドラインのチャンクが全て切り捨てられていた
# 対応：ガイドライン検索を添付文書検索より先に実行するよう順番を変更

question = "心血管疾患を合併した2型糖尿病患者に推奨される薬剤は何ですか"
section_numbers, _, is_guideline = detect_sections(question, llm)
drug_names = detect_drug_names(question, llm)

retrieved_docs = []

# セクション検索
if section_numbers:
    for num in section_numbers:
        results = vectorstore_c.similarity_search(
            question,
            k=2,
            filter={
                "$and": [
                    {"doc_type": {"$eq": "package_insert"}},
                    {"section_number": {"$eq": str(num)}}
                ]
            }
        )
        retrieved_docs.extend(results)

# ガイドライン検索
if is_guideline:
    guideline_results = vectorstore_c.similarity_search(
        question,
        k=3,
        filter={"doc_type": {"$eq": "guideline"}}
    )
    retrieved_docs.extend(guideline_results)

# 重複除去
seen = set()
unique_docs = []
for doc in retrieved_docs:
    content_hash = doc.page_content[:100]
    if content_hash not in seen:
        seen.add(content_hash)
        unique_docs.append(doc)

unique_docs = unique_docs[:5]

print(f"取得チャンク数: {len(unique_docs)}")
for doc in unique_docs:
    print(f"\n→ {doc.metadata.get('file_name')} section={doc.metadata.get('section_number')}")
    print(f"  {doc.page_content[:150]}")

取得チャンク数: 5

→ metformin.pdf section=1
  ○2型糖尿病
     ただし、下記のいずれかの治療で十分な効果が得られない
場合に限る。
    ⑴食事療法・運動療法のみ
    ⑵食事療法・運動療法に加えてスルホニルウレア剤を使用
   ○ 多嚢胞性卵巣症候群における排卵誘発、多嚢胞性卵巣症候
群の生殖補助医療における調節卵巣刺激
     

→ glimepiride.pdf section=1
  1．警告
重篤かつ遷延性の低血糖を起こすことがある。用法及び用
量、使用上の注意に特に留意すること。［8．1、11．1．1 参照］

→ insulin_aspart.pdf section=2
  2.禁忌（次の患者には投与しないこと） 
2.1 低血糖症状を呈している患者［11.1.1参照］ 
2.2 本剤の成分に対し過敏症の既往歴のある患者

→ insulin_glargine.pdf section=2
  2．禁忌（次の患者には投与しないこと）
2．1 低血糖症状を呈している患者［11．1．1 参照］
2．2 本剤の成分又は他のインスリン　グラルギン製剤に対し
過敏症の既往歴のある患者

→ linagliptin.pdf section=8
  8. 重要な基本的注意
8.1 本剤の使用にあたっては、患者に対し低血糖症状及びその対
処方法について十分説明すること。［9.1.1、11.1.1参照］
8.2 急性膵炎があらわれることがあるので、持続的な激しい腹痛、
嘔吐等の初期症状があらわれた場合には、速やかに医師の診察
を受けるよう患者に指導
